# **RAG:**

### Indexing phase:


In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

#document loader
loader = PyPDFLoader("C:/Aswath\RAG_Complete_Guide/embedded-systems.pdf")
docs = loader.load()

#text splitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap= 50,
    
)

chunks = splitter.split_documents(docs)
print(f"Created {len(chunks)} chunks")

# citation:

for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"]=i
    chunk.metadata["source_file"]=chunk.metadata.get("source", "unknown")

# embedding :
embed = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",    # free, 384 dimensions
    model_kwargs={"device": "cpu"} 
)

vector_db= Chroma.from_documents(
    documents=chunks,
    embedding=embed,
    persist_directory="./chroma.db" # saves to disk!

)
vector_db.persist()
print("Indexed and saved to ./chroma_db")  




C:\Users\aswat\AppData\Local\Temp\ipykernel_11624\98506548.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Aswath\RAG_Complete_Guide\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Created 1218 chunks


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7904.22it/s]


Indexed and saved to ./chroma_db


C:\Users\aswat\AppData\Local\Temp\ipykernel_11624\98506548.py:39: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_db.persist()


### Querying:

In [12]:
from langchain_groq import ChatGroq
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:

{context}

Question: {question}

Answer:
""")

load_dotenv()
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.1,
)



retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k":5}
)



chain = ({"context": retriever, "question": RunnablePassthrough()}
    |prompt
    | llm
    | StrOutputParser()
)

response = chain.invoke("What is the embedded system?")

print(response)


The provided context does not give a direct definition of an embedded system. However, it lists several characteristics that distinguish embedded systems from desktop systems:

1. Embedded systems often have power constraints.
2. Embedded systems often must operate under extreme environmental conditions.
3. Embedded systems have far fewer system resources than desktop systems.
4. Embedded systems often store all their object code in ROM.
5. Embedded systems require specialized tools and methods to be efficiently designed.
6. Embedded microprocessors often have dedicated debugging circuitry.

These characteristics imply that an embedded system is a specialized computer system designed to perform specific tasks, often in environments with limited resources and under particular constraints, such as power, space, or operating conditions.
